In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../marketing-campaigns.sqlite")

In [2]:
# Marketing Campaign Performance Analysis

## 1. Data Quality Checks
## 2. Channel Performance
## 3. Conversion Funnel
## 4. Conversion Type Analysis
## 5. Campaign Performance
## 6. Objective Performance
## 7. Monthly Performance
## 8. Budget Utilization
## 9. Lead Quality Analysis
## 10. Lead Quality by Channel

In [3]:
query = """
WITH spend_summary AS (
    SELECT
        campaign_id,
        SUM(spend) AS total_spend,
        SUM(impressions) AS impressions,
        SUM(clicks) AS clicks
    FROM daily_spend
    GROUP BY campaign_id
),

lead_summary AS (
    SELECT
        campaign_id,
        COUNT(DISTINCT lead_id) AS total_leads,
        ROUND(AVG(lead_score), 2) AS avg_lead_score
    FROM leads
    GROUP BY campaign_id
),

conversion_summary AS (
    SELECT
        l.campaign_id,

        COUNT(DISTINCT cv.conversion_id) AS total_conversions,

        COUNT(DISTINCT CASE
            WHEN cv.conversion_type = 'purchase'
            THEN cv.conversion_id
        END) AS purchases,

        COUNT(DISTINCT CASE
            WHEN cv.conversion_type = 'trial'
            THEN cv.conversion_id
        END) AS trials,

        SUM(
            CASE
                WHEN cv.conversion_type = 'purchase'
                THEN cv.revenue
                ELSE 0
            END
        ) AS total_revenue

    FROM leads l

    LEFT JOIN conversions cv
        ON l.lead_id = cv.lead_id

    GROUP BY l.campaign_id
)

SELECT
    c.campaign_id,
    c.campaign_name,
    c.channel,
    c.objective,
    c.start_date,
    c.end_date,
    c.budget,

    ROUND(s.total_spend, 2) AS total_spend,
    s.impressions,
    s.clicks,

    l.total_leads,
    l.avg_lead_score,

    COALESCE(cv.total_conversions, 0) AS total_conversions,
    COALESCE(cv.purchases, 0) AS purchases,
    COALESCE(cv.trials, 0) AS trials,
    ROUND(COALESCE(cv.total_revenue, 0), 2) AS total_revenue,

    ROUND(
        100.0 * s.total_spend / c.budget,
        2
    ) AS budget_utilization,

    ROUND(
        100.0 * s.clicks / s.impressions,
        2
    ) AS ctr,

    ROUND(
        100.0 * cv.purchases / l.total_leads,
        2
    ) AS purchase_rate,

    ROUND(
        cv.total_revenue / s.total_spend,
        2
    ) AS roas,

    ROUND(
        s.total_spend / cv.purchases,
        2
    ) AS cost_per_purchase,

    ROUND(
        cv.total_revenue / l.total_leads,
        2
    ) AS revenue_per_lead

FROM campaigns c

JOIN spend_summary s
    ON c.campaign_id = s.campaign_id

LEFT JOIN lead_summary l
    ON c.campaign_id = l.campaign_id

LEFT JOIN conversion_summary cv
    ON c.campaign_id = cv.campaign_id

ORDER BY total_revenue DESC;
"""

campaign_performance = pd.read_sql(query, conn)

campaign_performance.head()

,campaign_id,campaign_name,channel,objective,start_date,end_date,budget,total_spend,impressions,clicks,...,total_conversions,purchases,trials,total_revenue,budget_utilization,ctr,purchase_rate,roas,cost_per_purchase,revenue_per_lead
0,CMP0060,Affiliate 2024 060,affiliate,acquisition,2024-11-10,2025-04-05,79884.92,62309.32,6923196,174518,...,768,326,442,417738.92,78.00,2.52,6.39,6.70,191.13,81.85
1,CMP0050,Affiliate 2023 050,affiliate,awareness,2023-08-23,2023-12-27,37882.58,35219.40,3913208,96110,...,659,299,360,388919.88,92.97,2.46,7.32,11.04,117.79,95.18
2,CMP0021,Paid Search 2024 021,paid_search,awareness,2024-07-16,2024-11-30,76193.25,71337.50,3963132,137678,...,691,291,400,380892.89,93.63,3.47,6.16,5.34,245.15,80.58
3,CMP0061,Paid Search 2023 061,paid_search,lead_gen,2023-06-05,2023-10-14,50880.95,48907.03,2716997,96240,...,688,283,405,369829.93,96.12,3.54,6.68,7.56,172.82,87.24
4,CMP0081,Paid Search 2023 081,paid_search,awareness,2023-06-27,2023-11-15,48241.24,43017.04,2389775,83585,...,605,278,327,361820.39,89.17,3.50,6.70,8.41,154.74,87.21


In [4]:
campaign_performance.shape

(120, 22)

In [5]:
campaign_performance.isna().sum()

campaign_id           0
campaign_name         0
channel               0
objective             0
start_date            0
end_date              0
budget                0
total_spend           0
impressions           0
clicks                0
total_leads           0
avg_lead_score        0
total_conversions     0
purchases             0
trials                0
total_revenue         0
budget_utilization    0
ctr                   0
purchase_rate         0
roas                  0
cost_per_purchase     0
revenue_per_lead      0
dtype: int64

In [6]:
campaign_performance[
    ["total_spend", "total_revenue", "purchases", "total_leads"]
].sum()

total_spend       4523050.02
total_revenue    21561839.01
purchases           16691.00
total_leads        261684.00
dtype: float64

In [7]:
channel_performance = campaign_performance.groupby(
    "channel",
    as_index=False
).agg(
    total_spend=("total_spend", "sum"),
    total_revenue=("total_revenue", "sum"),
    total_leads=("total_leads", "sum"),
    purchases=("purchases", "sum"),
    budget=("budget", "sum")
)

channel_performance["roas"] = (
    channel_performance["total_revenue"]
    / channel_performance["total_spend"]
).round(2)

channel_performance["purchase_rate"] = (
    100 * channel_performance["purchases"]
    / channel_performance["total_leads"]
).round(2)

channel_performance["cost_per_purchase"] = (
    channel_performance["total_spend"]
    / channel_performance["purchases"]
).round(2)

channel_performance

,channel,total_spend,total_revenue,total_leads,purchases,budget,roas,purchase_rate,cost_per_purchase
0,affiliate,857835.07,5610700.73,67849,4332,1025491.41,6.54,6.38,198.02
1,display,955297.00,3734117.43,45149,2871,1138158.37,3.91,6.36,332.74
2,paid_search,897091.53,5439329.35,65268,4204,1079299.66,6.06,6.44,213.39
3,paid_social,984731.88,4690137.28,57619,3646,1180823.70,4.76,6.33,270.09
4,video,828094.54,2087554.22,25799,1638,994869.02,2.52,6.35,505.55


In [8]:
query = """
WITH spend AS (
    SELECT
        strftime('%Y-%m', spend_date) AS month,
        SUM(spend) AS total_spend
    FROM daily_spend
    GROUP BY month
),

revenue AS (
    SELECT
        strftime('%Y-%m', converted_at) AS month,
        SUM(
            CASE
                WHEN conversion_type = 'purchase'
                THEN revenue
                ELSE 0
            END
        ) AS total_revenue
    FROM conversions
    GROUP BY month
)

SELECT
    s.month,
    ROUND(s.total_spend, 2) AS total_spend,
    ROUND(COALESCE(r.total_revenue, 0), 2) AS total_revenue,
    ROUND(
        COALESCE(r.total_revenue, 0) / s.total_spend,
        2
    ) AS roas

FROM spend s

LEFT JOIN revenue r
    ON s.month = r.month

ORDER BY s.month;
"""

monthly_performance = pd.read_sql(query, conn)

monthly_performance

,month,total_spend,total_revenue,roas
0,2023-01,23686.55,52800.58,2.23
1,2023-02,49461.61,170806.55,3.45
2,2023-03,104283.65,393351.94,3.77
3,2023-04,104393.90,474970.42,4.55
4,2023-05,97734.10,435288.12,4.45
5,2023-06,123064.68,569578.22,4.63
6,2023-07,116810.98,699206.04,5.99
7,2023-08,134065.09,672633.67,5.02
8,2023-09,155294.61,884815.77,5.70
9,2023-10,171832.62,939852.38,5.47


In [9]:
query = """
SELECT
    CASE
        WHEN l.lead_score <= 25 THEN 'Low'
        WHEN l.lead_score <= 50 THEN 'Medium'
        WHEN l.lead_score <= 75 THEN 'High'
        ELSE 'Very High'
    END AS lead_quality,

    COUNT(DISTINCT l.lead_id) AS total_leads,

    COUNT(DISTINCT CASE
        WHEN cv.conversion_type = 'purchase'
        THEN cv.conversion_id
    END) AS purchases,

    ROUND(
        SUM(
            CASE
                WHEN cv.conversion_type = 'purchase'
                THEN cv.revenue
                ELSE 0
            END
        ),
        2
    ) AS total_revenue

FROM leads l

LEFT JOIN conversions cv
    ON l.lead_id = cv.lead_id

GROUP BY lead_quality

ORDER BY
    CASE lead_quality
        WHEN 'Low' THEN 1
        WHEN 'Medium' THEN 2
        WHEN 'High' THEN 3
        WHEN 'Very High' THEN 4
    END;
"""

lead_quality = pd.read_sql(query, conn)

lead_quality["purchase_rate"] = (
    100 * lead_quality["purchases"]
    / lead_quality["total_leads"]
).round(2)

lead_quality["revenue_per_lead"] = (
    lead_quality["total_revenue"]
    / lead_quality["total_leads"]
).round(2)

lead_quality

,lead_quality,total_leads,purchases,total_revenue,purchase_rate,revenue_per_lead
0,Low,65231,2080,2670778.95,3.19,40.94
1,Medium,65307,3476,4501825.40,5.32,68.93
2,High,65696,4881,6255811.73,7.43,95.22
3,Very High,65450,6254,8133422.93,9.56,124.27


In [12]:
campaign_performance.to_csv(
    "../data/processed/campaign_performance.csv",
    index=False
)

channel_performance.to_csv(
    "../data/processed/channel_performance.csv",
    index=False
)

monthly_performance.to_csv(
    "../data/processed/monthly_performance.csv",
    index=False
)

lead_quality.to_csv(
    "../data/processed/lead_quality.csv",
    index=False
)

In [13]:
import os

db_path = "../marketing-campaigns.sqlite"

print(f"Database size: {os.path.getsize(db_path) / (1024**2):.2f} MB")

Database size: 14.63 MB
